# Baseline Model

Train baseline LightGBM and XGBoost models, compare CV metrics, and generate a first submission.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
import lightgbm as lgb
import xgboost as xgb
import warnings

warnings.filterwarnings("ignore")

train = pd.read_csv("../data/processed/train_processed.csv")
test = pd.read_csv("../data/processed/test_processed.csv")

DROP = ["record_id", "flood_risk_score"]
X = train.drop(columns=DROP)
y = train["flood_risk_score"]
X_test = test.drop(columns=["record_id"])

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (20886, 59)
y shape: (20886,)


In [2]:
def competition_metric_proxy(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    balanced_error = (mae + rmse) / 2

    r2 = r2_score(y_true, y_pred)
    ev_penalty = max(0, 1 - r2)

    score = balanced_error * (1 + ev_penalty)
    return score

print("Metric defined")

Metric defined


In [3]:
def evaluate_model(model, X, y, n_splits=5, model_name="Model"):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    scores = []
    rmse_scores = []
    r2_scores = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        preds = np.clip(preds, 0, 1)

        score = competition_metric_proxy(y_val, preds)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        r2 = r2_score(y_val, preds)

        scores.append(score)
        rmse_scores.append(rmse)
        r2_scores.append(r2)

        print(
            f"  Fold {fold+1}: metric={score:.4f}  "
            f"rmse={rmse:.4f}  r2={r2:.4f}"
        )

    print(f"\n{model_name} Summary:")
    print(f"  Metric : {np.mean(scores):.4f} ± {np.std(scores):.4f}")
    print(f"  RMSE   : {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")
    print(f"  R²     : {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}")

    return np.mean(scores), np.mean(rmse_scores), np.mean(r2_scores)

In [4]:
lgbm_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

print("=== LightGBM Baseline ===")
lgbm_score, lgbm_rmse, lgbm_r2 = evaluate_model(
    lgbm_model, X, y, model_name="LightGBM"
)

=== LightGBM Baseline ===
  Fold 1: metric=0.4332  rmse=0.2416  r2=-0.0262
  Fold 2: metric=0.4330  rmse=0.2432  r2=-0.0163
  Fold 3: metric=0.4222  rmse=0.2370  r2=-0.0111
  Fold 4: metric=0.4336  rmse=0.2429  r2=-0.0183
  Fold 5: metric=0.4341  rmse=0.2414  r2=-0.0328

LightGBM Summary:
  Metric : 0.4312 ± 0.0045
  RMSE   : 0.2412 ± 0.0022
  R²     : -0.0209 ± 0.0076


In [5]:
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

print("=== XGBoost Baseline ===")
xgb_score, xgb_rmse, xgb_r2 = evaluate_model(
    xgb_model, X, y, model_name="XGBoost"
)

=== XGBoost Baseline ===
  Fold 1: metric=0.4327  rmse=0.2416  r2=-0.0261
  Fold 2: metric=0.4316  rmse=0.2428  r2=-0.0137
  Fold 3: metric=0.4227  rmse=0.2373  r2=-0.0131
  Fold 4: metric=0.4295  rmse=0.2419  r2=-0.0104
  Fold 5: metric=0.4288  rmse=0.2399  r2=-0.0196

XGBoost Summary:
  Metric : 0.4291 ± 0.0035
  RMSE   : 0.2407 ± 0.0020
  R²     : -0.0166 ± 0.0056


In [6]:
results = pd.DataFrame(
    {
        "model": ["LightGBM", "XGBoost"],
        "metric": [lgbm_score, xgb_score],
        "rmse": [lgbm_rmse, xgb_rmse],
        "r2": [lgbm_r2, xgb_r2],
    }
)

print(results.sort_values("metric"))

import os

os.makedirs("../results", exist_ok=True)
results.to_csv("../results/baseline_scores.csv", index=False)

      model    metric      rmse        r2
1   XGBoost  0.429057  0.240712 -0.016594
0  LightGBM  0.431216  0.241227 -0.020943


In [7]:
best_model = lgbm_model

best_model.fit(X, y)
preds = best_model.predict(X_test)
preds = np.clip(preds, 0, 1)

submission = pd.DataFrame(
    {"record_id": test["record_id"], "flood_risk_score": preds}
)

import os

os.makedirs("../submissions", exist_ok=True)
submission.to_csv("../submissions/sub_baseline_lgbm.csv", index=False)

print("Submission saved")
print(submission["flood_risk_score"].describe().round(4))
print("\nSample:")
print(submission.head())

Submission saved
count    5300.0000
mean        0.4774
std         0.0651
min         0.2429
25%         0.4335
50%         0.4758
75%         0.5204
max         0.7476
Name: flood_risk_score, dtype: float64

Sample:
  record_id  flood_risk_score
0   F104559          0.385141
1   F100765          0.361831
2   F107573          0.493936
3   F110345          0.524413
4   F118850          0.622626


In [8]:
log_entry = pd.DataFrame(
    [
        {
            "date": pd.Timestamp.today().date(),
            "member": "your_name",
            "model": "lgbm_baseline",
            "cv_metric": round(lgbm_score, 4),
            "cv_rmse": round(lgbm_rmse, 4),
            "cv_r2": round(lgbm_r2, 4),
            "lb_score": None,
            "notes": "first baseline submission",
        }
    ]
)

log_path = "../results/scores_log.csv"
if os.path.exists(log_path):
    log = pd.read_csv(log_path)
    log = pd.concat([log, log_entry], ignore_index=True)
else:
    log = log_entry

log.to_csv(log_path, index=False)
print("Score logged")

Score logged


In [9]:
# If mean prediction beats your model, that's the baseline to beat
mean_pred = np.full(len(y), y.mean())

print("=== Mean Prediction Baseline ===")
print(f"Metric : {competition_metric_proxy(y, mean_pred):.4f}")
print(f"RMSE   : {np.sqrt(mean_squared_error(y, mean_pred)):.4f}")
print(f"R²     : {r2_score(y, mean_pred):.4f}")
print(f"y mean : {y.mean():.4f}")

=== Mean Prediction Baseline ===
Metric : 0.4228
RMSE   : 0.2388
R²     : 0.0000
y mean : 0.4780


In [10]:
# What if we just predict the district mean for each row?
train_raw = pd.read_csv('../data/raw/train.csv')
test_raw  = pd.read_csv('../data/raw/test.csv')

district_mean_map = train_raw.groupby('district')['flood_risk_score'].mean()
global_mean       = train_raw['flood_risk_score'].mean()

district_preds_train = train_raw['district'].map(district_mean_map)\
                                            .fillna(global_mean)

print("=== District Mean Prediction ===")
print(f"Metric : {competition_metric_proxy(y, district_preds_train):.4f}")
print(f"RMSE   : {np.sqrt(mean_squared_error(y, district_preds_train)):.4f}")
print(f"R²     : {r2_score(y, district_preds_train):.4f}")

# Generate submission with just district means
district_preds_test = test_raw['district'].map(district_mean_map)\
                                          .fillna(global_mean)
district_preds_test = np.clip(district_preds_test, 0, 1)

sub_district = pd.DataFrame({
    'record_id'       : test_raw['record_id'],
    'flood_risk_score': district_preds_test
})
sub_district.to_csv('../submissions/sub_district_mean.csv', index=False)
print("\nDistrict mean submission saved")

=== District Mean Prediction ===
Metric : 0.4147
RMSE   : 0.2367
R²     : 0.0174

District mean submission saved


In [11]:
# We dropped place_name earlier (792 unique values)
# But with 20k rows it might have enough coverage to target encode

place_counts = train_raw['place_name'].value_counts()
print("place_name coverage:")
print(f"  Total unique    : {train_raw['place_name'].nunique()}")
print(f"  Appears 5+ times: {(place_counts >= 5).sum()}")
print(f"  Appears 10+ times: {(place_counts >= 10).sum()}")
print(f"  Appears 20+ times: {(place_counts >= 20).sum()}")

# Target encode place_name for frequent ones only
place_mean = train_raw.groupby('place_name')['flood_risk_score'].mean()
place_count= train_raw.groupby('place_name')['flood_risk_score'].count()

# Only use places with 10+ occurrences
reliable_places = place_count[place_count >= 10].index
place_mean_reliable = place_mean[reliable_places]

train_place_preds = train_raw['place_name'].map(place_mean_reliable)\
                                           .fillna(global_mean)

print("\n=== Place Name Mean Prediction ===")
print(f"Metric : {competition_metric_proxy(y, train_place_preds):.4f}")
print(f"RMSE   : {np.sqrt(mean_squared_error(y, train_place_preds)):.4f}")
print(f"R²     : {r2_score(y, train_place_preds):.4f}")

place_name coverage:
  Total unique    : 792
  Appears 5+ times: 792
  Appears 10+ times: 791
  Appears 20+ times: 722

=== Place Name Mean Prediction ===
Metric : 0.4079
RMSE   : 0.2342
R²     : 0.0380


## Geographic Hierarchical Encoding

In [12]:
train_raw = pd.read_csv("../data/raw/train.csv")
test_raw = pd.read_csv("../data/raw/test.csv")

global_mean = train_raw["flood_risk_score"].mean()
global_std = train_raw["flood_risk_score"].std()

place_stats = (
    train_raw.groupby("place_name")["flood_risk_score"]
    .agg(["mean", "std", "count"])
    .rename(columns={"mean": "place_mean", "std": "place_std", "count": "place_count"})
)

district_stats = (
    train_raw.groupby("district")["flood_risk_score"]
    .agg(["mean", "std", "count"])
    .rename(
        columns={
            "mean": "district_mean",
            "std": "district_std",
            "count": "district_count",
        }
    )
)

print("Place stats sample:")
print(place_stats.sort_values("place_mean", ascending=False).head(10))
print("\nDistrict stats:")
print(district_stats.sort_values("district_mean", ascending=False))

Place stats sample:
                   place_mean  place_std  place_count
place_name                                           
Kiriwatta East       0.666322   0.267340           23
Nayagama West        0.620654   0.292108           26
Udarakanda South     0.610915   0.225694           27
Nayavila South       0.600112   0.228129           24
Mahapitiya North     0.598665   0.274891           20
Kirimedura East      0.592410   0.276015           21
Kirimedura North     0.592276   0.240809           25
Polawatta South      0.586119   0.197811           31
Welimedura South     0.584284   0.209864           19
Galathota Central    0.583263   0.226145           30

District stats:
              district_mean  district_std  district_count
district                                                 
Vavuniya           0.538169      0.231435             775
Monaragala         0.533144      0.229661             781
Mullaitivu         0.514518      0.239042             802
Badulla            0.5130

In [13]:
def bayesian_smooth(stats_df, mean_col, count_col, global_mean, k=10):
    smoothed = (
        stats_df[count_col] * stats_df[mean_col] + k * global_mean
    ) / (stats_df[count_col] + k)
    return smoothed

place_stats["place_smoothed"] = bayesian_smooth(
    place_stats, "place_mean", "place_count", global_mean, k=10
)
district_stats["district_smoothed"] = bayesian_smooth(
    district_stats, "district_mean", "district_count", global_mean, k=50
)

print("Smoothed place means (top 10):")
print(place_stats["place_smoothed"].sort_values(ascending=False).head(10))

Smoothed place means (top 10):
place_name
Kiriwatta East       0.609267
Nayagama West        0.581040
Udarakanda South     0.575003
Nayavila South       0.564210
Polawatta South      0.559759
Kirimedura North     0.559638
Mahapitiya North     0.558457
Galathota Central    0.556958
Welivila             0.556739
Kirimedura East      0.555517
Name: place_smoothed, dtype: float64


In [14]:
def add_geo_features(df, place_stats, district_stats, global_mean, global_std):
    df = df.copy()

    df["place_smoothed"] = df["place_name"].map(
        place_stats["place_smoothed"]
    ).fillna(global_mean)
    df["place_std"] = df["place_name"].map(place_stats["place_std"]).fillna(
        global_std
    )
    df["place_count"] = df["place_name"].map(place_stats["place_count"]).fillna(0)

    df["district_smoothed"] = df["district"].map(
        district_stats["district_smoothed"]
    ).fillna(global_mean)
    df["district_std"] = df["district"].map(district_stats["district_std"]).fillna(
        global_std
    )

    df["place_district_diff"] = df["place_smoothed"] - df["district_smoothed"]

    return df

train_geo = add_geo_features(
    train_raw, place_stats, district_stats, global_mean, global_std
)
test_geo = add_geo_features(
    test_raw, place_stats, district_stats, global_mean, global_std
)

print("Geo features added")
print(
    train_geo[["place_smoothed", "district_smoothed", "place_district_diff"]]
    .describe()
    .round(4)
)

Geo features added
       place_smoothed  district_smoothed  place_district_diff
count      20886.0000         20886.0000           20886.0000
mean           0.4781             0.4769               0.0012
std            0.0337             0.0296               0.0444
min            0.3652             0.4138              -0.1603
25%            0.4567             0.4594              -0.0294
50%            0.4783             0.4780               0.0006
75%            0.4997             0.4995               0.0303
max            0.6093             0.5345               0.1955


In [15]:
GEO_FEATURES = [
    "place_smoothed",
    "district_smoothed",
    "place_std",
    "place_count",
    "place_district_diff",
]

BEST_ORIGINAL = [
    "inundation_area_sqm",
    "latitude",
    "longitude",
    "terrain_roughness_index",
    "extreme_weather_index",
    "infrastructure_score",
    "seasonal_index",
    "socioeconomic_status_index",
    "elevation_m",
    "distance_to_river_m",
    "rainfall_7d_mm",
    "drainage_index",
    "ndwi",
    "historical_flood_count",
    "population_density_per_km2",
]

ENCODED_CATS = [
    "flood_occurrence_current_event",
    "water_presence_flag",
    "urban_rural",
    "road_quality",
    "electricity",
    "water_supply",
]

ALL_FEATURES = GEO_FEATURES + BEST_ORIGINAL + ENCODED_CATS

binary_map = {
    "flood_occurrence_current_event": {"Yes": 1, "No": 0},
    "is_good_to_live": {"Yes": 1, "No": 0},
    "water_presence_flag": {"Likely": 1, "Unlikely": 0},
    "urban_rural": {"Urban": 1, "Rural": 0},
}
road_map = {"Good (paved)": 3, "Fair": 2, "Poor (unpaved)": 1, "No road access": 0}
electricity_map = {"Grid": 2, "Mixed": 1, "Off-grid (solar)": 0}
water_map = {
    "Municipal": 4,
    "Tube-well": 3,
    "Well": 2,
    "Surface water": 1,
    "Rainwater harvesting": 0,
}

for df in [train_geo, test_geo]:
    for col, mapping in binary_map.items():
        if col in df.columns:
            df[col] = df[col].map(mapping)
    if "road_quality" in df.columns:
        df["road_quality"] = df["road_quality"].map(road_map)
    if "electricity" in df.columns:
        df["electricity"] = df["electricity"].map(electricity_map)
    if "water_supply" in df.columns:
        df["water_supply"] = df["water_supply"].map(water_map)

X_geo = train_geo[ALL_FEATURES].fillna(train_geo[ALL_FEATURES].median())
y_geo = train_raw["flood_risk_score"]
X_test_geo = test_geo[ALL_FEATURES].fillna(train_geo[ALL_FEATURES].median())

print("Feature set shape:", X_geo.shape)

Feature set shape: (20886, 26)


In [16]:
lgbm_geo = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

print("=== LightGBM with Geo Features ===")
geo_score, geo_rmse, geo_r2 = evaluate_model(
    lgbm_geo, X_geo, y_geo, model_name="LightGBM Geo"
)

print("\nImprovement over mean baseline:")
print(f"  Metric : {0.4228 - geo_score:+.4f}")
print(f"  R²     : {geo_r2 - 0.0:.4f}")

=== LightGBM with Geo Features ===
  Fold 1: metric=0.4117  rmse=0.2354  r2=0.0257
  Fold 2: metric=0.4168  rmse=0.2386  r2=0.0216
  Fold 3: metric=0.3993  rmse=0.2303  r2=0.0458
  Fold 4: metric=0.4097  rmse=0.2359  r2=0.0393
  Fold 5: metric=0.4097  rmse=0.2343  r2=0.0272

LightGBM Geo Summary:
  Metric : 0.4094 ± 0.0057
  RMSE   : 0.2349 ± 0.0027
  R²     : 0.0319 ± 0.0091

Improvement over mean baseline:
  Metric : +0.0134
  R²     : 0.0319


In [17]:
lgbm_geo.fit(X_geo, y_geo)
geo_preds = lgbm_geo.predict(X_test_geo)
geo_preds = np.clip(geo_preds, 0, 1)

sub_geo = pd.DataFrame(
    {"record_id": test_raw["record_id"], "flood_risk_score": geo_preds}
)
sub_geo.to_csv("../submissions/sub_lgbm_geo.csv", index=False)
print("Geo submission saved")
print(sub_geo["flood_risk_score"].describe().round(4))

Geo submission saved
count    5300.0000
mean        0.4784
std         0.0730
min         0.2314
25%         0.4299
50%         0.4753
75%         0.5251
max         0.7749
Name: flood_risk_score, dtype: float64
